# Sosso Trajet — Nettoyage & Feature Engineering

**Hackathon DATA 4 CHANGE | Projet 3 | Talent Tally × MP3**

Ce notebook applique la stratégie de nettoyage validée et enrichit le dataset Yango de features pour l'entraînement du modèle ML de prédiction de prix.

**Entrée** : (7 341 lignes × 11 colonnes)  
**Sortie** : (~7 173 lignes × 21 colonnes)


## 1. Imports & chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [ ]:
# Charger le dataset brut
df = pd.read_csv('SossoTrajet_Dataset.csv')
n0 = len(df)
print(f"Chargement initial : {n0} lignes x {df.shape[1]} colonnes")
df.head()

In [ ]:
# Aperçu rapide
print("Types:")
print(df.dtypes)
print("\nValeurs manquantes:")
print(df.isnull().sum())

In [ ]:
# Apercu general : distributions des variables numeriques + valeurs manquantes
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

df['distance'].plot(kind='hist', bins=50, ax=axes[0,0], color='#42A5F5', edgecolor='white')
axes[0,0].set_title("Distance (km)")

df['duree_estimee'].plot(kind='hist', bins=50, ax=axes[0,1], color='#66BB6A', edgecolor='white')
axes[0,1].set_title("Duree estimee (min)")

df['prix_eco'].plot(kind='hist', bins=50, ax=axes[0,2], color='#FFA726', edgecolor='white')
axes[0,2].set_title("Prix Eco (FCFA)")

df['prix_confort'].plot(kind='hist', bins=50, ax=axes[1,0], color='#AB47BC', edgecolor='white')
axes[1,0].set_title("Prix Confort (FCFA)")

df['prix_confort_plus'].plot(kind='hist', bins=50, ax=axes[1,1], color='#EF5350', edgecolor='white')
axes[1,1].set_title("Prix Confort+ (FCFA)")

missing = df.isnull().sum()
missing[missing > 0].plot(kind='bar', ax=axes[1,2], color='#FF7043', edgecolor='white')
axes[1,2].set_title("Valeurs manquantes par colonne")
axes[1,2].set_ylabel("Nombre de NaN")

plt.suptitle("Apercu general du dataset brut", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot pour detecter les valeurs aberrantes (avant nettoyage)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df[['distance', 'duree_estimee']], ax=axes[0])
axes[0].set_title("Distance & Duree — Avant nettoyage")

sns.boxplot(data=df[['prix_eco', 'prix_confort', 'prix_confort_plus']], ax=axes[1])
axes[1].set_title("Prix — Avant nettoyage (FCFA)")

plt.suptitle("Detection des valeurs aberrantes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution par ville et disponibilite des vehicules
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['ville'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3', '#FF9800'], edgecolor='white')
axes[0].set_title("Nombre de trajets par ville")
axes[0].set_ylabel("Nombre de trajets")
axes[0].tick_params(axis='x', rotation=0)

df['disponibilite_vehicules'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title("Disponibilite des vehicules")
axes[1].set_ylabel("Nombre de trajets")

plt.tight_layout()
plt.show()

## 2. Phase A — Nettoyage (10 étapes)

Ordre d'exécution : corrections > imputations > suppressions.
Les corrections/imputations sont faites avant les suppressions pour ne pas perdre de lignes récupérables.

### A1 — Suppression des doublons stricts

**Justification** : vérification défensive. Aucun effet attendu sur ce dataset.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"A1 drop_duplicates : {before - len(df)} lignes retirees -> {len(df)} lignes")

### A3 — Correction de `heure_plage == '24h–25h'`

**Justification** : bug d'encodage manifeste (24h = 0h). On corrige plutôt que de supprimer.  
*(A3 avant A4 pour que l'imputation n'ait plus à gérer cette valeur.)*

In [ ]:
n_corrige = (df['heure_plage'] == '24h–25h').sum()
df.loc[df['heure_plage'] == '24h–25h', 'heure_plage'] = '00h–01h'
print(f"A3 correction 24h-25h : {n_corrige} ligne(s) corrigee(s)")

### A4 — Imputation de `heure_plage == 'inconnu'` par le mode + colonne flag

**Justification** : garder les 27 lignes, imputer avec la plage la plus fréquente (biais minimal), ajouter un flag pour traçabilité.

In [ ]:
# Calcul du mode (sur les valeurs valides uniquement)
mode_heure = df.loc[df['heure_plage'] != 'inconnu', 'heure_plage'].mode().iloc[0]

# Création du flag AVANT imputation
df['heure_plage_imputed'] = (df['heure_plage'] == 'inconnu').astype(int)
n_imp = df['heure_plage_imputed'].sum()

# Imputation
df.loc[df['heure_plage'] == 'inconnu', 'heure_plage'] = mode_heure
print(f"A4 imputation 'inconnu' : {n_imp} ligne(s) imputee(s) avec mode='{mode_heure}' + flag 'heure_plage_imputed' ajoute")

### A2 — Imputation de `duree_estimee` manquante par la médiane

**Justification** : la durée est la feature la plus prédictive (corr 0.72 avec le prix). Ne pas perdre de lignes. Médiane calculée hors outliers (>120 min) pour robustesse.

In [ ]:
mediane_duree = df.loc[
    (df['duree_estimee'].notna()) & (df['duree_estimee'] <= 120),
    'duree_estimee'
].median()

n_imp_duree = df['duree_estimee'].isna().sum()
df['duree_estimee'] = df['duree_estimee'].fillna(mediane_duree)
print(f"A2 imputation duree_estimee : {n_imp_duree} NaN imputes avec mediane = {mediane_duree:.1f} min")

### A5 — Suppression des `distance <= 0.1 km`

**Justification** : 137 lignes avec distance=0 mais départ ≠ arrivée = erreurs de géoloc Yango. Distance est feature obligatoire, pas d'imputation possible sans fuite.

In [ ]:
before = len(df)
df = df[df['distance'] > 0.1].reset_index(drop=True)
print(f"A5 suppression distance <= 0.1 : {before - len(df)} lignes retirees -> {len(df)} lignes")

### A6 — Conservation des distances > 100 km (choix assumé)

**Justification** : 4 lignes conservées (dont une à 4 481 km). L'impact sur le modèle est neutralisé par la transformation `log_distance` en Phase B.

In [ ]:
n_big = (df['distance'] > 100).sum()
print(f"A6 distance > 100 km : {n_big} ligne(s) conservee(s) — neutralisees par log_distance")

### A7 — Suppression des `duree_estimee > 120 min`

**Justification** : 99.9e percentile à ~80 min. Au-delà de 120, ce sont des bugs Yango (compteur qui continue).

In [ ]:
before = len(df)
df = df[df['duree_estimee'] <= 120].reset_index(drop=True)
print(f"A7 suppression duree > 120 min : {before - len(df)} lignes retirees -> {len(df)} lignes")

### A8 — Suppression des `prix_eco < 100` FCFA

**Justification** : prix à 2, 3, 6 FCFA = économiquement impossible (minimum réel ~300 FCFA). On garde les NaN pour l'instant (gestion en Phase B).

In [ ]:
before = len(df)
df = df[(df['prix_eco'].isna()) | (df['prix_eco'] >= 100)].reset_index(drop=True)
print(f"A8 suppression prix_eco < 100 : {before - len(df)} lignes retirees -> {len(df)} lignes")

### A9 — Suppression des `prix_eco > 10 000` FCFA

**Justification** : au-dela de 10 000 FCFA, les valeurs sont incohérentes (ex: 54 450 FCFA pour 2 min). Les 5 000-10 000 FCFA correspondent à de vrais longs trajets et sont conservés.

In [ ]:
before = len(df)
df = df[(df['prix_eco'].isna()) | (df['prix_eco'] <= 10000)].reset_index(drop=True)
print(f"A9 suppression prix_eco > 10000 : {before - len(df)} lignes retirees -> {len(df)} lignes")

### A10 — Prix incohérents entre catégories → NaN

**Justification** : quand `prix_confort < prix_eco` ou `prix_confort_plus < prix_confort`, on met la valeur fautive à NaN (on ne supprime pas la ligne). Cohérent avec la stratégie "1 modèle sur Éco + dérivation par ratio".

In [ ]:
# Cas 1 : prix_confort < prix_eco
mask_c = (df['prix_confort'].notna()
          & df['prix_eco'].notna()
          & (df['prix_confort'] < df['prix_eco']))

# Cas 2 : prix_confort_plus < prix_confort
mask_cp = (df['prix_confort_plus'].notna()
           & df['prix_confort'].notna()
           & (df['prix_confort_plus'] < df['prix_confort']))

df.loc[mask_c, 'prix_confort'] = np.nan
df.loc[mask_cp, 'prix_confort_plus'] = np.nan
print(f"A10 incoherences -> NaN : {mask_c.sum()} confort, {mask_cp.sum()} confort+")
print(f"\nBilan Phase A : {len(df)} lignes ({100*len(df)/n0:.1f}% conservees)")

In [ ]:
# Boxplot apres nettoyage Phase A
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df[['distance', 'duree_estimee']], ax=axes[0], color='#66BB6A')
axes[0].set_title("Distance & Duree — Apres Phase A")

sns.boxplot(data=df[['prix_eco', 'prix_confort', 'prix_confort_plus']], ax=axes[1], color='#42A5F5')
axes[1].set_title("Prix — Apres Phase A (FCFA)")

plt.suptitle("Dataset apres nettoyage (Phase A)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlation sur donnees nettoyees (Phase A)
plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes(include=['number']).corr(),
            annot=True, cmap='RdYlGn', fmt='.2f')
plt.title("Matrice de correlation — Donnees nettoyees (apres Phase A)")
plt.tight_layout()
plt.show()

## 3. Phase B — Feature engineering (9 colonnes ajoutées)

Ces colonnes transforment le dataset nettoyé en un jeu de features prêt pour l'entraînement.

### B1 — `heure_num` (entier 0-23)

**Justification** : transformer 24 catégories en numérique continu pour capturer la proximité entre heures voisines.

In [ ]:
df['heure_num'] = df['heure_plage'].str.split('h').str[0].astype(int)
print(df['heure_num'].describe())

In [ ]:
# Distribution des plages horaires par ville
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, ville in zip(axes, ['Douala', 'Yaounde']):
    sub = df[df['ville'] == ville]
    sub['heure_num'].value_counts().sort_index().plot(kind='bar', ax=ax, color='teal', edgecolor='white')
    ax.set_title(f"Plages horaires — {ville}")
    ax.set_xlabel("Heure de debut")
    ax.set_ylabel("Nombre de trajets")

plt.tight_layout()
plt.show()

### B2 — `is_pointe` (binaire)

**Justification** : heures 7-9h et 17-19h. Effet prix +7.5% selon l'EDA. Variable interprétable pour le jury.

In [ ]:
df['is_pointe'] = df['heure_num'].isin([7, 8, 9, 17, 18, 19]).astype(int)
print(f"Trajets en heure de pointe : {df['is_pointe'].sum()} ({100*df['is_pointe'].mean():.1f}%)")

In [ ]:
# Prix moyen en heure de pointe vs heure normale
prix_pointe = df.groupby('is_pointe')['prix_eco'].mean()
fig, ax = plt.subplots(figsize=(7, 5))
prix_pointe.plot(kind='bar', ax=ax, color=['#42A5F5', '#EF5350'], edgecolor='white')
ax.set_xticklabels(['Heure normale', 'Heure de pointe'], rotation=0)
ax.set_title("Prix Eco moyen : pointe vs normale")
ax.set_ylabel("Prix moyen (FCFA)")
for i, v in enumerate(prix_pointe):
    ax.text(i, v + 20, f"{v:.0f} FCFA", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### B3 — `is_nuit` (binaire)

**Justification** : 22h-5h. Profil de trajet différent (moins de congestion, tarification parfois spécifique).

In [ ]:
df['is_nuit'] = ((df['heure_num'] >= 22) | (df['heure_num'] <= 5)).astype(int)
print(f"Trajets de nuit : {df['is_nuit'].sum()} ({100*df['is_nuit'].mean():.1f}%)")

### B4 — `dispo_ordinal` (entier 0-3)

**Justification** : la disponibilité a une relation monotone avec le prix (vert 281 → rouge 390 FCFA/km). L'encodage ordinal préserve cet ordre. `gris` (info non mesurée) est mappé à 0 car prix moyen proche de vert.

In [ ]:
dispo_map = {'vert': 0, 'jaune': 1, 'orange': 2, 'rouge': 3}
df['dispo_ordinal'] = df['disponibilite_vehicules'].map(dispo_map).fillna(0).astype(int)
print(df.groupby('disponibilite_vehicules')['dispo_ordinal'].first())

In [ ]:
# Prix median par niveau de disponibilite
dispo_labels = {0: 'vert', 1: 'jaune', 2: 'orange', 3: 'rouge'}
colors = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['prix_eco', 'prix_confort', 'prix_confort_plus']):
    medians = df.groupby('dispo_ordinal')[col].median()
    medians.index = [dispo_labels.get(i, str(i)) for i in medians.index]
    medians.plot(kind='bar', ax=ax, color=colors[:len(medians)], edgecolor='white')
    ax.set_title(f"Prix median ({col}) par disponibilite")
    ax.set_ylabel("FCFA")
    ax.set_xlabel("")

plt.suptitle("Impact de la disponibilite sur les prix", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### B5 — `ville_encoded` (binaire)

**Justification** : 1 si Yaoundé, 0 sinon. Permet au modèle d'apprendre un léger décalage tarifaire sans surapprendre (Douala marginale à 119 lignes).

In [ ]:
df['ville_encoded'] = (df['ville'] == 'Yaounde').astype(int)
print(df.groupby('ville')['ville_encoded'].first())

### B6 — `vitesse_kmh` (float, capée à 120)

**Justification** : distance / (durée / 60). Révèle la congestion implicite. Cap à 120 km/h pour éliminer les divisions par ~0.

In [ ]:
df['vitesse_kmh'] = df['distance'] / (df['duree_estimee'] / 60)
df['vitesse_kmh'] = df['vitesse_kmh'].clip(upper=120)
print(df['vitesse_kmh'].describe())

In [ ]:
# Distribution de la vitesse calculee
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['vitesse_kmh'].plot(kind='hist', bins=50, ax=axes[0], color='#AB47BC', edgecolor='white')
axes[0].set_title("Distribution — Vitesse (km/h)")
axes[0].set_xlabel("km/h")

sns.boxplot(data=df[['vitesse_kmh']], ax=axes[1], color='#AB47BC')
axes[1].set_title("Boxplot — Vitesse (km/h)")

plt.tight_layout()
plt.show()

### B7 — `log_distance` et `log_duree`

**Justification** : relation prix-distance log-linéaire (effet seuil + forfait minimum). Log stabilise la variance et **neutralise les outliers de distance conservés en A6**.

In [ ]:
df['log_distance'] = np.log1p(df['distance'])
df['log_duree'] = np.log1p(df['duree_estimee'])
print(df[['log_distance', 'log_duree']].describe())

In [ ]:
# Comparaison distribution brute vs log-transformee
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['distance'].plot(kind='hist', bins=50, ax=axes[0,0], color='#42A5F5', edgecolor='white')
axes[0,0].set_title("Distance brute (km)")

df['log_distance'].plot(kind='hist', bins=50, ax=axes[0,1], color='#1565C0', edgecolor='white')
axes[0,1].set_title("log_distance (apres log1p)")

df['duree_estimee'].plot(kind='hist', bins=50, ax=axes[1,0], color='#66BB6A', edgecolor='white')
axes[1,0].set_title("Duree brute (min)")

df['log_duree'].plot(kind='hist', bins=50, ax=axes[1,1], color='#2E7D32', edgecolor='white')
axes[1,1].set_title("log_duree (apres log1p)")

plt.suptitle("Effet de la log-transformation sur les distributions", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### B8 — `log_prix_eco` (target transformée)

**Justification** : distribution prix_eco asymétrique à droite. Entraîner sur log(prix) uniformise l'erreur sur toute la gamme → MAE réduite de 10-15%. Back-transformation via `expm1` à l'inférence.

In [ ]:
df['log_prix_eco'] = np.log1p(df['prix_eco'])
print(df['log_prix_eco'].describe())

## 4. Validation du dataset final

In [ ]:
print(f"Shape final : {df.shape}")
print(f"\nColonnes : {list(df.columns)}")
print(f"\nValeurs manquantes restantes :")
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Corrélations avec la target
num_features = ['distance', 'duree_estimee', 'heure_num', 'is_pointe',
                'is_nuit', 'dispo_ordinal', 'ville_encoded', 'vitesse_kmh',
                'log_distance', 'log_duree']
corr = df[num_features + ['prix_eco']].corr()['prix_eco'].drop('prix_eco').sort_values(ascending=False)
print("Correlations avec prix_eco :")
print(corr.round(3))

In [ ]:
# Heatmap complete des features engineerees
num_features = ['distance', 'duree_estimee', 'heure_num', 'is_pointe',
                'is_nuit', 'dispo_ordinal', 'ville_encoded', 'vitesse_kmh',
                'log_distance', 'log_duree', 'prix_eco']
plt.figure(figsize=(12, 9))
sns.heatmap(df[num_features].corr(), annot=True, cmap='RdYlGn', fmt='.2f')
plt.title("Matrice de correlation — Features finales vs Prix")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter : prix_eco vs log_distance (par ville)
plt.figure(figsize=(10, 6))
for ville, color in [('Douala', '#2196F3'), ('Yaounde', '#FF9800')]:
    sub = df[df['ville'] == ville].dropna(subset=['prix_eco'])
    plt.scatter(sub['log_distance'], sub['prix_eco'],
                alpha=0.3, s=8, label=ville, color=color)
plt.xlabel("log_distance")
plt.ylabel("Prix Eco (FCFA)")
plt.title("Prix Eco vs log_distance (par ville)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ratios entre catégories (pour la dérivation au moment du ML)
sub = df.dropna(subset=['prix_eco', 'prix_confort', 'prix_confort_plus'])
ratio_c = (sub['prix_confort'] / sub['prix_eco']).median()
ratio_cp = (sub['prix_confort_plus'] / sub['prix_eco']).median()
print(f"Ratio Confort / Eco  (mediane) : {ratio_c:.3f}")
print(f"Ratio Confort+ / Eco (mediane) : {ratio_cp:.3f}")

## 5. Export du dataset nettoyé

In [ ]:
df.to_csv('SossoTrajet_Clean.csv', index=False)
print(f"Dataset exporte : SossoTrajet_Clean.csv")
print(f"   {len(df)} lignes x {df.shape[1]} colonnes")